In [21]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import adi
import time
import binascii

In [22]:
def bin_to_text(binary_arr):
    bin_str = "".join(str(bit) for bit in binary_arr)
    text = ""
    for i in range(0, len(bin_str), 7):
        byte = bin_str[i:i+7]
        if len(byte) == 7 :
          text += chr(int(byte, 2))
    return text

def text_to_bin(text):
    temp_bin_str = ''.join(format(ord(c), '07b') for c in text)
    bin_data = np.array([int(bit) for bit in temp_bin_str], dtype=int)
    return bin_data

def verify_crc(received_frame):
    received_payload = received_frame[:-16]
    received_crc = received_frame[-16:]
    byte_data = np.packbits(received_payload)  
    crc = binascii.crc_hqx(byte_data, 0xFFFF)  
    expected_crc = np.array(list(np.binary_repr(crc, width=16)), dtype=np.uint8)  
    if np.array_equal(received_crc, expected_crc):
        return True  
    else:
        return False 

def append_crc_to_frame(payload):
    byte_data = np.packbits(payload)  
    crc = binascii.crc_hqx(byte_data, 0xFFFF)  
    crc_bits = np.array(list(np.binary_repr(crc, width=16)), dtype=np.uint8)  
    return np.concatenate((payload, crc_bits))

def binary_array_to_int(binary_array):
    return int("".join(map(str, binary_array)), 2)


def int_to_fixed_binary_array(n, bits=8):
    return [int(bit) for bit in format(n, f'0{bits}b')]

In [23]:
sample_rate = 10e6 # Hz
carrier_freq = 980e6 # Hz
num_samps = 100000 

sdr2 = adi.Pluto("ip:192.168.3.1")
sdr2.sample_rate = int(sample_rate)

sdr2.rx_lo = int(carrier_freq)
sdr2.rx_rf_bandwidth = int(sample_rate)
sdr2.rx_buffer_size = num_samps
sdr2.gain_control_mode_chan0 = 'slow_attack'
# sdr2.rx_hardwaregain_chan0 = 70 # dB

sdr2.tx_rf_bandwidth = int(sample_rate) 
sdr2.tx_lo = int(carrier_freq)
sdr2.tx_hardwaregain_chan0 = -15
sdr2.tx_cyclic_buffer = False 

In [24]:
final_text = ''
count = 0
while(count < 10) :   
    sdr2.rx_enabled = True
    for _ in range(10):
        _ = sdr2.rx() 
    print("RX buffer cleared")
    rx_samples = sdr2.rx()
    energy = np.sum(np.abs(rx_samples)**2)
    avg_power = energy / len(rx_samples)
    if avg_power > 200000:
        t3 = time.time()
        print('decoding start')
        t1 =time.time()
        sps = 8
        num_taps = 101
        beta = 0.34
        Ts = sps 
        t = np.arange(num_taps) - (num_taps-1)//2
        rc_pulse = np.sinc(t/Ts) * np.cos(np.pi*beta*t/Ts) / (1 - (2*beta*t/Ts)**2)
        matched_filter =  rc_pulse
        MF_op = np.convolve(rx_samples, matched_filter, mode='same')
        max_allowed = 1.5
        rx_max = np.max(np.abs(MF_op))
        scale_coarse = max_allowed / rx_max
        normalized_rx = MF_op * scale_coarse
        squared_FO = normalized_rx**2
        fs = sample_rate
        psd = np.fft.fftshift(np.abs(np.fft.fft(squared_FO)))
        f = np.linspace(-fs/2.0, fs/2.0, len(psd))
        max_freq = f[np.argmax(psd)]
        Ts = 1/fs
        t = np.arange(0, Ts*len(normalized_rx), Ts) # create time vector
        coarse_corrected = normalized_rx * np.exp(-1j*2*np.pi*max_freq*t/2.0)
        power = np.mean(np.abs(coarse_corrected)**2)
        normalized_2 = coarse_corrected / np.sqrt(power)
        samples = normalized_2
        N = len(samples)
        phase = 0
        freq = 0
        alpha = 0.132
        beta = 0.00932
        fine_correct = np.zeros(N, dtype=np.complex64)
        freq_log = []
        for i in range(N):
            fine_correct[i] = samples[i] * np.exp(-1j*phase)
            error = np.real(fine_correct[i]) * np.imag(fine_correct[i]) 
            freq += (beta * error)
            freq_log.append(freq * fs / (2*np.pi)) 
            phase += freq + (alpha * error)
            while phase >= 2*np.pi:
                phase -= 2*np.pi
            while phase < 0:
                phase += 2*np.pi
        samples  = fine_correct[100:]
        samples_interpolated = signal.resample_poly(samples, 16, 1)
        mu = 0 
        out = np.zeros(len(samples) + 10, dtype=np.complex64)
        out_rail = np.zeros(len(samples) + 10, dtype=np.complex64) #
        i_in = 0 
        i_out = 2 
        while i_out < len(samples) and i_in+16 < len(samples):
            out[i_out] = samples_interpolated[i_in*16 + int(mu*16)]
            out_rail[i_out] = int(np.real(out[i_out]) > 0) + 1j*int(np.imag(out[i_out]) > 0)
            x = (out_rail[i_out] - out_rail[i_out-2]) * np.conj(out[i_out-1])
            y = (out[i_out] - out[i_out-2]) * np.conj(out_rail[i_out-1])
            mm_val = np.real(y - x)
            mu += sps + 0.3*mm_val
            i_in += int(np.floor(mu))
            mu = mu - np.floor(mu) 
            i_out += 1 
        out = out[2:i_out] 
        time_sync = out
        barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
        barker_correlated = np.correlate(time_sync, barker_code, mode='full')
        offset  = np.argmax(np.abs(barker_correlated))+1
        peak = np.real(barker_correlated)[offset-1]
        if peak < 0:
            time_sync = -1*time_sync
        payload_size = 280
        n_bits = payload_size + 16 + 10
        fr_sync_op = time_sync[offset:offset+n_bits]
        recovered_bits = (fr_sync_op > 0).astype(int)
        received_frame = recovered_bits
        if verify_crc(received_frame):
            frame_id = binary_array_to_int(received_frame[:10])
            print('count', count, 'frame_id', frame_id)
            if frame_id!= count:
                count+= 1
            # print('frame_id', frame_id)
            decoded_text = bin_to_text(recovered_bits[10:payload_size+10])
            print('decoding end')
            final_text+= decoded_text
            print('Decoded text:', decoded_text)
            # print(time.time()-t1)
            time.sleep(2)
            print(time.time() - t3)
            ###########################################
            sdr2.rx_enabled = False
            sdr2.tx_enabled = True
            time.sleep(0.05)
            t_start = time.time()
            t_end = 0
            print('transmitting ACK start')
            text = 'ACKACKACKACKACKACKACKACKACKACK'
            message_bits = text_to_bin(text)
            frame_no = int_to_fixed_binary_array(frame_id, 10)
            payload = np.concatenate((frame_no, message_bits))
            data_bits = append_crc_to_frame(payload)
            bpsk_symbols = np.array([-1 if bit == 0 else 1 for bit in data_bits])
            barker_code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
            barker_appended_bpsk = np.concatenate((barker_code,  bpsk_symbols))
            sps = 8
            zero_padded = np.zeros(len(barker_appended_bpsk) * sps)
            zero_padded[::sps] = barker_appended_bpsk
            num_taps = 101
            beta = 0.34
            Ts = sps
            t = np.arange(num_taps) - (num_taps-1)//2
            rc_pulse = np.sinc(t/Ts) * np.cos(np.pi*beta*t/Ts) / (1 - (2*beta*t/Ts)**2)
            ps_conv_output = np.convolve(zero_padded, rc_pulse, mode = 'full')
            pulse_shaped = ps_conv_output[num_taps//2: -1-num_taps//2]
            tx_signal = pulse_shaped * (2**14)
            while(t_end < 3):
                t_end = time.time() - t_start
                sdr2.tx(tx_signal)
            sdr2.tx_destroy_buffer()
            sdr2.tx_enabled = False
            print('transmitting ACK end')

            print(time.time() - t3)
            # print(time.time()-t1)
        else:
            time.sleep(5.5)

print(final_text)

RX buffer cleared
RX buffer cleared
RX buffer cleared
decoding start
RX buffer cleared
RX buffer cleared
RX buffer cleared
RX buffer cleared
RX buffer cleared
RX buffer cleared
RX buffer cleared
RX buffer cleared
decoding start
count 0 frame_id 1
decoding end
Decoded text: Mental health is vital to overall well-b
2.4563937187194824
transmitting ACK start
transmitting ACK end
5.509108543395996
RX buffer cleared
decoding start
count 1 frame_id 2
decoding end
Decoded text: eing, influencing how we think, feel, an
2.4440195560455322
transmitting ACK start
transmitting ACK end
5.49651026725769
RX buffer cleared
decoding start
RX buffer cleared
decoding start
count 2 frame_id 3
decoding end
Decoded text: d act. 
It shapes relationships, decisio
2.45324444770813
transmitting ACK start
transmitting ACK end
5.505417585372925
RX buffer cleared
decoding start
count 3 frame_id 4
decoding end
Decoded text: ns, and coping abilities. Yet, stigma of
2.4482510089874268
transmitting ACK start
transmitti

KeyboardInterrupt: 